from the previous hyperparameter tuning notebook, three models were tested and these were the results:

|model|Mean RMSE|STD RMSE|
|-----|---------|--------|
|Ridge|0.136576|0.023301|
|RandomForest|0.142412|0.008456|
|LightGBM|0.130961|0.0104067|

from the results, LightGBM has the lowest average RMSE, but it's variance is not the best. In this notebook I will analyse the errors made by LightGBM and its variance, and also its interpretability, to get better information on the final model selection

# Setup and Loading

In [8]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

In [9]:
import numpy as np
import pandas as pd
import joblib

from src.data import load_train_data
from src.features import add_engineered_features

data = load_train_data()
X = add_engineered_features(data)
y = np.log1p(data['SalePrice'])

MODELS_DIR = ROOT / "models"
ridge_pipe = joblib.load(MODELS_DIR/ "ridge_final.pkl")
lgb_pipe = joblib.load(MODELS_DIR / "lgbm_final.pkl")
rfr_pipe = joblib.load(MODELS_DIR /"rfr_final.pkl")

In [10]:
# produce Out-Of-Fold predictions
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import mean_squared_error


cv = KFold(n_splits=5, shuffle= True, random_state=42)
ridge_pred = cross_val_predict(ridge_pipe, X, y, cv=cv, method='predict', n_jobs=-1)
lgb_pred = cross_val_predict(lgb_pipe, X, y, cv=cv, method='predict', n_jobs=-1)
rfr_pred = cross_val_predict(rfr_pipe, X, y, cv=cv, method='predict', n_jobs=-1)

ridge_rmse = np.sqrt(mean_squared_error(y, ridge_pred))
lgb_rmse = np.sqrt(mean_squared_error(y, lgb_pred))
rfr_rmse = np.sqrt(mean_squared_error(y, rfr_pred))

print('Ridge OOF RMSE:', ridge_rmse)
print('LGB OOF RMSE:', lgb_rmse)
print('RFR OOF RMSE:', rfr_rmse)

Ridge OOF RMSE: 0.14701048328624955
LGB OOF RMSE: 0.13685615941445264
RFR OOF RMSE: 0.1429946318942553
